# Sentiment Classification on Game Reviews Demonstration

Author: Qian Ying Wong

The notebook loads a **fine-tuned BERT (bert-base-uncased) model with warmup ratio = 0.1** and runs predictions on a smaller dataset (N=3000). Below we list out all the models explored by our team.

Note that due to runtime and the sake of simplicity when TAs run this notebook, project.ipynb only includes the demonstration of one variation of BERT.

## All Models (for your reference)
### Baseline Models:
- TF-IDF + Logistic Regression
- Bag of Words + Logistic Regression
- Weighted BoW + Logistic Regression
- Static Embeddings + Logistic Regression
### Advanced Models:
- Fine-tuned BERT (bert-base-uncased) with different hyperparameters*
  - learning rate: 1e-5
  - learning rate: 3e-5
  - epochs: 4
  - sequence length: 128
  - warmup ratio: 0.1 (achived best accuracy)
- Fine-tuned BERT (bert-large-cased)
### Benchmark Model:
- Qwen2.5-7B-Instruct
  - zero shot
  - one shot

Total runtime should be around 1 minute on T4 GPU using Google Colab.

*More details in the written final report


### 1. Install dependencies and import libraries

In [29]:
!pip -q install huggingface_hub transformers scikit-learn pandas torch datasets joblib

In [30]:
import pandas as pd
import torch
import joblib

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
from huggingface_hub import hf_hub_download, login
from sklearn.metrics import accuracy_score
from datasets import load_dataset

### 2. Load testing dataset
- the entire Kaggle dataset has been uploaded to HuggingFace
- subset of 3000 datapoints from our Kaggle dataset
   - we chose 3000 to keep runtime short and sweet

In [31]:
dataset = load_dataset("QYW03/cs175_testing_dataset")
test_dataset = dataset["train"]  # huggingface automatically assigns "train"
# because I only uploaded one dataset; to clarify this is not training data,
# just testing data, but to grab it from huggingface I had to name it like this
subset = test_dataset.shuffle(seed=42).select(range(3000))
X_test = subset["review_content"]
y_test = [1 if x else 0 for x in subset["is_positive"]]

### 3. Load BERT
- BERT "bert-base-uncased" with warmup ratio=0.1
- this model has been pretrained by Qian Ying Wong and uploaded to her HuggingFace account

In [32]:
# login()  # I hope I did everything correctly and you wouldn't have to login
# to run the model, but just in case,
# you can still un-comment this cell and enter an access token

In [33]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo = "QYW03/cs175-finetuned-base-bert"

tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForSequenceClassification.from_pretrained(repo)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.


In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Using device:", device)

Using device: cuda


Function that helps run batch inference:

In [35]:
def predict_batch(texts, model, tokenizer, device, batch_size=32):
    preds = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        encodings = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=256,
            return_tensors="pt"
        )

        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = model(**encodings)
            batch_preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()

        preds.extend(batch_preds)

    return preds

### 4. Run predictions

In [36]:
bert_preds = predict_batch(X_test, model, tokenizer, device)

bert_acc = accuracy_score(y_test, bert_preds)
print(f"BERT accuracy on 3000-sample subset: {bert_acc:.4f}")

BERT accuracy on 3000-sample subset: 0.8450


### 5. Some analysis

Classification report:

In [37]:
print(classification_report(
    y_test,
    bert_preds,
    target_names=["negative", "positive"],
    digits=4
))

              precision    recall  f1-score   support

    negative     0.8645    0.8292    0.8465      1546
    positive     0.8260    0.8618    0.8435      1454

    accuracy                         0.8450      3000
   macro avg     0.8452    0.8455    0.8450      3000
weighted avg     0.8458    0.8450    0.8450      3000



Dataframe with true labels:

In [38]:
results_df = pd.DataFrame({
    "review_content": X_test,
    "true_label": y_test,
    "pred_label": bert_preds
})

results_df["true_label_name"] = results_df["true_label"].map({0: "negative", 1: "positive"})
results_df["pred_label_name"] = results_df["pred_label"].map({0: "negative", 1: "positive"})
results_df["correct"] = results_df["true_label"] == results_df["pred_label"]

results_df.head(10)

,review_content,true_label,pred_label,true_label_name,pred_label_name,correct
0,Imagine first...Leaft 4 Dead with aliens inste...,1,1,positive,positive,True
1,One of the scientist models looks like Gus Fri...,1,1,positive,positive,True
2,Give us back the old dota,0,0,negative,negative,True
3,You will find less cheaters on Tinder,0,1,negative,positive,False
4,"If you just want to try it out, it's okay. How...",0,0,negative,negative,True
5,Almost great game but versus community are so ...,0,1,negative,positive,False
6,"You can play as germans, which is cool.\n\nBut...",1,1,positive,positive,True
7,best puzzle game by far. good work valve,1,1,positive,positive,True
8,Roll back the latest patch for Templar Assassi...,0,0,negative,negative,True
9,Nice just pls make good antichat,1,1,positive,positive,True


In [39]:
print("=== Correctly classified examples ===")

correct_examples = results_df[results_df["correct"]].head(5)

for i, row in correct_examples.iterrows():
    print(f"\nExample {i}")
    print("True:", row["true_label_name"])
    print("Pred:", row["pred_label_name"])
    print("Review:", row["review_content"][:500])

=== Correctly classified examples ===

Example 0
True: positive
Pred: positive
Review: Imagine first...Leaft 4 Dead with aliens instead.....oooOOo that rhymed.
Now switch the view from first person to Top-Down view in the style of an RTS.
Now add 4 Classes, each with specific roles and requirements and equipment for each mission.
Then add an SDK download for the community to make as mach content as they want.
...What you get is a Overhead Alien shooter Survival action game with a large amount of Customisation.

Addictive, Gorgeous....free.

Example 1
True: positive
Pred: positive
Review: One of the scientist models looks like Gus Fring. This is the most interesting part of Blue Shift.

Example 2
True: negative
Pred: negative
Review: Give us back the old dota

Example 4
True: negative
Pred: negative
Review: If you just want to try it out, it's okay. However, if you want to experience it in the long run, it's better to play the original or Black Mesa instead. There are a lot of bugs and 

In [40]:
print("=== Misclassified examples ===")

wrong_examples = results_df[~results_df["correct"]].head(5)

for i, row in wrong_examples.iterrows():
    print(f"\nExample {i}")
    print("True:", row["true_label_name"])
    print("Pred:", row["pred_label_name"])
    print("Review:", row["review_content"][:500])

=== Misclassified examples ===

Example 3
True: negative
Pred: positive
Review: You will find less cheaters on Tinder

Example 5
True: negative
Pred: positive
Review: Almost great game but versus community are so fucking toxic Just play normal campaign or non versus game mode if you don't want to encounter or hate/don't like toxic people. But I still like playing this game.

Example 14
True: positive
Pred: negative
Review: I am a 15 second year old father, probably one of the oldest peopel playing cs 1.6. Im a single father to a 95 year old son. He got this game for beiram from his 40 year old grand grandma. I never talked to him much but since he bought the game even less. He has 20055 hours in the game. I wish his 20 year old mother never bought it for Sabbat

Example 17
True: negative
Pred: positive
Review: one of the best games but im sick of it

Example 19
True: positive
Pred: negative
Review: It's boring in a fun way


## Takeaway
This notebook demonstrates inference with our fine-tuned BERT sentiment classifier on a 3000-example subset of our Kaggle testing dataset. The BERT model uses "bert-base-uncased", with hyperparameter configuration of warmup ratio = 0.1. The reason we chose this variation of BERT was because it was the highest performing version among all the different configurations of "bert-base-uncased" we experimented on.

Finallu, the notebook also reports prediction accuracy and shows examples of both successful and failed classifications to illustrate model behavior.